In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error

## Preparacion de datos

In [4]:
df = pd.read_csv('../data/train/transformed_train.csv')

# Convert the 'Moneda' column to binary (1 for 'US$', 0 for '$')
df['Moneda'] = df['Moneda'].apply(lambda x: 1 if x == 'US$' else 0)
df.loc[df['Moneda'] == 1, 'Precio'] = df.loc[df['Moneda'] == 1, 'Precio'] * 1100

# Convert Kilometers column to numeric, removing 'km' if present
df['Kilómetros'] = df['Kilómetros'].str.replace(' km', '', regex=False).str.replace('km', '', regex=False)
df['Kilómetros'] = pd.to_numeric(df['Kilómetros'], errors='coerce')

# Handle the price conversion for US$ values (Moneda = 1)
# Assuming 1 US$ = 1100 local currency

def extraer_formato_correcto(texto):
    if isinstance(texto, str):
        match = re.search(r'\b(\d\.\d)', texto, re.IGNORECASE)
        return match.group(1) if match else np.nan
    return np.nan

# Aplicar solo sobre la columna 'Motor'
df['Motor'] = df['Motor'].apply(extraer_formato_correcto).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14603 entries, 0 to 14602
Data columns (total 40 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   idx                      14603 non-null  int64  
 1   Marca                    14603 non-null  object 
 2   Modelo                   14603 non-null  object 
 3   Versión                  14085 non-null  object 
 4   Título                   14603 non-null  object 
 5   Motor                    14439 non-null  float64
 6   Turbo                    14603 non-null  int64  
 7   cv                       7397 non-null   float64
 8   Tracción                 7197 non-null   object 
 9   Color                    14288 non-null  object 
 10  Tipo de combustible      14603 non-null  object 
 11  Puertas                  14603 non-null  float64
 12  Transmisión              14590 non-null  object 
 13  Con cámara de retroceso  3749 non-null   object 
 14  Kilómetros            

In [5]:
# Variable target
y = df["Precio"]
X = df.drop(columns=["idx", "Título", "Descripción", "Precio"])

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Target/mean encoding (usar media de train solamente)
mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means)  # aplicar los mismos valores (NaN si valor nuevo)

# Categóricas para one-hot
one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

# Normalizamos las columnas numéricas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])  # usar la misma escala

# Ordinal encode antes de KNN imputación
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  # usar misma codificación

# Imputación KNN solo con train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Volvemos a DataFrame
X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

# One-hot encoding (sin dummy_na)
X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

# Alineamos columnas (puede faltar alguna en val que está en train)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (11682, 61)
Val: (2921, 61)


In [6]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11682 entries, 12283 to 7270
Data columns (total 61 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Marca                        11682 non-null  float64
 1   Modelo                       11682 non-null  float64
 2   Versión                      11682 non-null  float64
 3   Motor                        11682 non-null  float64
 4   Turbo                        11682 non-null  float64
 5   cv                           11682 non-null  float64
 6   Color                        11682 non-null  float64
 7   Puertas                      11682 non-null  float64
 8   Kilómetros                   11682 non-null  float64
 9   Moneda                       11682 non-null  float64
 10  Año                          11682 non-null  float64
 11  desc_pca_1                   11682 non-null  float64
 12  desc_pca_2                   11682 non-null  float64
 13  desc_pca_3        

In [7]:
X_train.describe()

,Marca,Modelo,Versión,Motor,Turbo,cv,Color,Puertas,Kilómetros,Moneda,...,Con cámara de retroceso_1.0,Tracción_0.0,Tracción_0.2,Tracción_0.4,Tracción_0.6,Tracción_0.8,Tracción_1.0,Tipo de vendedor_0.0,Tipo de vendedor_1.0,Tipo de vendedor_2.0
count,1.168200e+04,1.168200e+04,11682.000000,11682.000000,1.168200e+04,11682.000000,11682.000000,1.168200e+04,1.168200e+04,1.168200e+04,...,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000
mean,8.332850e-17,3.938336e-17,-0.005590,0.001850,1.602705e-16,-0.121165,0.001071,6.082372e-19,-4.835486e-17,-7.755025e-17,...,0.576442,0.346602,0.083804,0.096816,0.088683,0.087228,0.296867,0.468242,0.309964,0.221794
std,1.000043e+00,1.000043e+00,0.985673,0.998712,1.000043e+00,0.831164,0.996299,1.000043e+00,1.000043e+00,1.000043e+00,...,0.494143,0.475908,0.277106,0.295719,0.284298,0.282181,0.456897,0.499012,0.462498,0.415471
min,-1.786287e+00,-1.488684e+00,-1.590005,-1.316338,-7.325987e-01,-2.511220,-4.914436,-1.684965e-02,-6.272061e-01,-7.915595e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-5.494088e-01,-5.924153e-01,-0.528697,-0.503525,-7.325987e-01,-0.626509,-0.021388,-1.304367e-02,-6.272061e-01,-7.915595e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-4.107608e-01,-2.937877e-01,-0.181328,-0.340962,-7.325987e-01,-0.350000,0.361347,-1.304367e-02,-6.672732e-02,-7.915595e-01,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.537983e-01,4.806843e-01,0.413343,0.309288,1.365004e+00,0.094957,0.657051,-1.304367e-02,3.348097e-01,1.263329e+00,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000
max,5.035542e+00,9.082448e+00,12.824015,7.462043,1.365004e+00,6.530941,23.713548,7.641989e+01,8.302634e+01,1.263329e+00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Base Line

In [8]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("Base Line")
mlflow.set_tracking_uri("http://localhost:5000")

In [9]:
reports = []

## Regresor lineal

In [10]:
# Train the model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict on the validation set
y_pred_train_lr = lr.predict(X_train)
y_pred_val_lr = lr.predict(X_val)

In [11]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y_train, y_pred_train, y_val, y_pred_val):
    # Calculate train metrics
    mse_train = mean_squared_error(y_train, y_pred_train)
    r2_train = r2_score(y_train, y_pred_train)
    mae_train = mean_absolute_error(y_train, y_pred_train)
    evs_train = explained_variance_score(y_train, y_pred_train)
    max_err_train = max_error(y_train, y_pred_train)
    medae_train = median_absolute_error(y_train, y_pred_train)
    
    # Calculate validation metrics
    mse_val = mean_squared_error(y_val, y_pred_val)
    r2_val = r2_score(y_val, y_pred_val)
    mae_val = mean_absolute_error(y_val, y_pred_val)
    evs_val = explained_variance_score(y_val, y_pred_val)
    max_err_val = max_error(y_val, y_pred_val)
    medae_val = median_absolute_error(y_val, y_pred_val)
    
    # Create a table format for metrics
    print("Metric       |    Train     |  Validation")
    print("-------------|--------------|---------------")
    print(f"MSE          |   {mse_train:.2e}   |   {mse_val:.2e}")
    print(f"R²           |   {r2_train:.4f}     |   {r2_val:.4f}")
    print(f"MAE          |   {mae_train:.2e}   |   {mae_val:.2e}")
    print(f"EVS          |   {evs_train:.4f}     |   {evs_val:.4f}")
    print(f"Max Error    |   {max_err_train:.2e}   |   {max_err_val:.2e}")
    print(f"Median AE    |   {medae_train:.2e}   |   {medae_val:.2e}")
    print()
    
    # Return metrics in a dictionary for reporting
    return {
        "mse_train": mse_train,
        "r2_train": r2_train,
        "mse_val": mse_val,
        "r2_val": r2_val,
        "mae_train": mae_train,
        "evs_train": evs_train,
        "max_err_train": max_err_train,
        "medae_train": medae_train,
        "mae_val": mae_val,
        "evs_val": evs_val,
        "max_err_val": max_err_val,
        "medae_val": medae_val
    }

In [12]:
# Calculate metrics for Linear Regression model
lr_metrics = calculate_metrics(
    y_train, y_pred_train_lr, 
    y_val, y_pred_val_lr,
)

lr_metrics['name'] = "Linear Regression"
lr_metrics["model"] = lr
lr_metrics["params"] = None  # No params for Linear Regression


# Add to reports list
reports.append(lr_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.30e+14   |   1.40e+14
R²           |   0.7377     |   0.7135
MAE          |   5.67e+06   |   5.95e+06
EVS          |   0.7377     |   0.7135
Max Error    |   5.63e+08   |   3.07e+08
Median AE    |   3.56e+06   |   3.62e+06



## Random Forest

In [13]:
# Definir los hiperparámetros en un diccionario
rf_params = {
    'n_estimators': 100,       # cantidad de árboles
    'max_depth': None,         # sin profundidad máxima (puede sobreajustar si no se regula)
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'random_state': 42,
    'n_jobs': -1               # usar todos los núcleos disponibles
}

In [14]:
# Crear el modelo Random Forest
rf_reg = RandomForestRegressor(**rf_params)
rf_reg.fit(X_train, y_train)

# Predecir en el set de validación
y_pred_train_rf = rf_reg.predict(X_train)
y_pred_val_rf = rf_reg.predict(X_val)


In [15]:
# Calcular métricas para el modelo Random Forest
rf_metrics = calculate_metrics(
    y_train, y_pred_train_rf,
    y_val, y_pred_val_rf,
)

rf_metrics["name"] = "Random Forest"
rf_metrics["model"] = rf_reg
rf_metrics["params"] = rf_params  # Agregar los parámetros del modelo al reporte

# Agregar a la lista de reportes
reports.append(rf_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.40e+13   |   5.50e+13
R²           |   0.9718     |   0.8873
MAE          |   1.08e+06   |   2.74e+06
EVS          |   0.9718     |   0.8873
Max Error    |   2.27e+08   |   2.40e+08
Median AE    |   5.19e+05   |   1.35e+06



## XGBoost

In [16]:
# Definir los parámetros en un diccionario
xgb_params = {
    'objective': 'reg:squarederror',  # objetivo para regresión
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.1,
    'random_state': 42
}

In [17]:
# Crear el modelo regresor con los parámetros
xgb_reg = XGBRegressor(**xgb_params)
xgb_reg.fit(X_train, y_train)

y_pred_train_xgb = xgb_reg.predict(X_train)
y_pred_val_xgb = xgb_reg.predict(X_val)


In [18]:
# Calcular métricas para el modelo XGBoost
xgb_metrics = calculate_metrics(
    y_train, y_pred_train_xgb,
    y_val, y_pred_val_xgb
)

xgb_metrics["name"] = "XGBoost"
xgb_metrics["model"] = xgb_reg
xgb_metrics["params"] = xgb_params  # Agregar los parámetros del modelo al reporte
# Agregar a la lista de reportes
reports.append(xgb_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   2.82e+13   |   4.71e+13
R²           |   0.9433     |   0.9035
MAE          |   2.90e+06   |   3.27e+06
EVS          |   0.9433     |   0.9035
Max Error    |   2.76e+08   |   1.59e+08
Median AE    |   1.86e+06   |   1.90e+06



## Red neuronal

In [19]:
# Definir los hiperparámetros en un diccionario
mlp_params = {
    'hidden_layer_sizes': (100, 50),   # dos capas ocultas con 100 y 50 neuronas
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,                   # L2 regularization
    'learning_rate': 'adaptive',
    'max_iter': 500,
    'random_state': 42
}

In [20]:
# Crear el modelo MLP
mlp_reg = MLPRegressor(**mlp_params)
mlp_reg.fit(X_train, y_train)

y_pred_train_mlp = mlp_reg.predict(X_train)
y_pred_val_mlp = mlp_reg.predict(X_val)

/home/elianaostro/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [21]:
# Calcular métricas para el modelo MLP
mlp_params_metrics = calculate_metrics(
    y_train, y_pred_train_mlp,
    y_val, y_pred_val_mlp
)

mlp_params_metrics["name"] = "MLP"
mlp_params_metrics["model"] = mlp_reg
mlp_params_metrics["params"] = mlp_params  # Agregar los parámetros del modelo al reporte

# Agregar a la lista de reportes
reports.append(mlp_params_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.26e+14   |   1.26e+14
R²           |   0.7465     |   0.7416
MAE          |   5.56e+06   |   5.85e+06
EVS          |   0.7466     |   0.7417
Max Error    |   5.56e+08   |   2.95e+08
Median AE    |   3.73e+06   |   3.96e+06



In [26]:
input_example = X_train.iloc[0:1]

for report in reports:
    with mlflow.start_run(run_name=report["name"]):
        if report["params"]:
            mlflow.log_params(report["params"])
        mlflow.log_metrics({
            "mse_val": report["mse_val"],
            "r2_val": report["r2_val"],
            "mae_val": report["mae_val"],
            "evs_val": report["evs_val"],
            "max_err_val": report["max_err_val"],
            "medae_val": report["medae_val"]
        })
        signature = infer_signature(input_example, report["model"].predict(input_example))
        if report["name"] == "XGBoost":
            mlflow.xgboost.log_model(report["model"], "model", input_example=input_example, signature=signature, metadata={"Data Source": "Transformed 2 Train Data"})
        else:
            mlflow.sklearn.log_model(report["model"], "model", input_example=input_example, signature=signature, metadata={"Data Source": "Transformed 2 Train Data"})

2025/06/28 19:47:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/28 19:47:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: http://localhost:5000/#/experiments/306040637862002389/runs/55aaca8d475248889b61e1b23bef9e99
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389


2025/06/28 19:47:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/306040637862002389/runs/61eb82b83b1c4b1da43ffa14413e8a55
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389


/home/elianaostro/miniconda3/lib/python3.12/site-packages/xgboost/sklearn.py:1028: UserWarning: [19:47:50] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2025/06/28 19:47:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://localhost:5000/#/experiments/306040637862002389/runs/3322237c54c043ed8f2d0fdccedfb813
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389
🏃 View run MLP at: http://localhost:5000/#/experiments/306040637862002389/runs/7c5be6f6e8844b10a01c697177689b2d
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389
